# Orange Fruit Quality Classification Using a Basic CNN

**Framework:** PyTorch
**Dataset:** Orange Fruit Image Dataset for Classification (Mendeley Data, 2026, CC BY 4.0) — 15,000 images, 5 classes (fresh, black-spotted, canker-affected, green, rotten)
DOI: `10.17632/zk4hzhtcdd.2` — https://data.mendeley.com/datasets/zk4hzhtcdd/2
**Input:** RGB images resized to 128 × 128 pixels

This notebook contains only:
1. Dataset loading and preprocessing
2. Training, validation, and testing split (stratified 70/15/15, fixed seed)
3. CNN architecture
4. Model training with loss and accuracy tracking
5. Training vs. validation accuracy and loss graphs
6. Final testing

To study the effect of a hyperparameter, change **one value at a time** in the *Hyperparameters* cell and rerun from that cell onward.

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Hyperparameters

**Change these values to run your experiments.** Change one at a time and keep the others fixed.

- `LEARNING_RATE`: size of each weight update step
- `BATCH_SIZE`: number of images used for each weight update
- `DROPOUT_RATE`: fraction of neurons switched off during training
- `EPOCHS`: number of full passes through the training set
- `USE_AUGMENTATION`: `False` = basic transform only, `True` = flips, rotation, brightness/contrast changes

In [ ]:
IMG_SIZE = 128
LEARNING_RATE = 0.001
BATCH_SIZE = 32
DROPOUT_RATE = 0.3
EPOCHS = 15
USE_AUGMENTATION = False

## 3. Dataset loading and preprocessing

In [ ]:
# Collect every image in the dataset. The class name is the name of the folder the image is in.
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_paths = []
image_classes = []
for current, subdirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(IMG_EXT):
            image_paths.append(os.path.join(current, f))
            image_classes.append(os.path.basename(current))

class_names = sorted(set(image_classes))
num_classes = len(class_names)
labels = np.array([class_names.index(c) for c in image_classes])

print("Total images:", len(image_paths))
print("Classes     :", class_names)
print("Per class   :", dict(zip(class_names, np.bincount(labels))))
assert 2 <= num_classes <= 10, "Unexpected number of classes. Check the folder names printed above."

In [ ]:
# Stratified split: 70% training, 15% validation, 15% testing (fixed seed)
rng = np.random.RandomState(42)
train_idx, val_idx, test_idx = [], [], []
for c in range(num_classes):
    idx = np.where(labels == c)[0]
    rng.shuffle(idx)
    n_train = int(0.70 * len(idx))
    n_val = int(0.15 * len(idx))
    train_idx += list(idx[:n_train])
    val_idx   += list(idx[n_train:n_train + n_val])
    test_idx  += list(idx[n_train + n_val:])

# Dataset that fetches one image at a time
class OrangeDataset(Dataset):
    def __init__(self, indices, transform):
        self.paths = [image_paths[i] for i in indices]
        self.labels = [int(labels[i]) for i in indices]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, ix):
        image = Image.open(self.paths[ix]).convert("RGB")
        return self.transform(image), self.labels[ix]

basic_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# Augmentation is used on the training images only (no hue/saturation change: the classes differ by colour)
aug_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
])

train_transform = aug_transform if USE_AUGMENTATION else basic_transform

train_data = OrangeDataset(train_idx, train_transform)
val_data   = OrangeDataset(val_idx,   basic_transform)
test_data  = OrangeDataset(test_idx,  basic_transform)

print("Training images  :", len(train_data))
print("Validation images:", len(val_data))
print("Testing images   :", len(test_data))

In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# quick look at one training image
img, label = train_data[0]
plt.figure(figsize=(2, 2))
plt.imshow(img.permute(1, 2, 0))
plt.title(class_names[label])
plt.axis("off")
plt.show()

## 4. CNN architecture

The images are RGB, so the first convolution layer takes **3 input channels**. The last layer has **5 output neurons**, one per class.

Four convolutional blocks (`Conv → ReLU → MaxPool`) followed by fully connected layers with dropout.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=5, dropout=0.3):
        super().__init__()

        # Convolutional layers
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),    # 3x128x128 -> 16x128x128
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 16x64x64

            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # 32x64x64
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 32x32x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # 64x32x32
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 64x16x16

            nn.Conv2d(64, 64, kernel_size=3, padding=1),   # 64x16x16
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 64x8x8
        )

        # Fully connected layers / classification head
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = CNNModel(num_classes, DROPOUT_RATE).to(device)
print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

## 5. Model training with loss and accuracy tracking

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc * 100:.2f}%")

## 6. Training vs. validation accuracy and loss

In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_acc"], label="Training Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.title("Training vs. Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_loss"], label="Training Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.title("Training vs. Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. Final testing

Run this after the training run you want to evaluate. The test set is not used for training or for choosing hyperparameters; it is evaluated only here.

In [ ]:
test_loss, test_acc = validate_epoch(model, test_loader, criterion, device)

print("Test images  :", len(test_data))
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.2f}%")